In [1]:
from csv import DictReader
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "pnf").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

data_path = root / "data" / "nifty.csv"
data_path

WindowsPath('e:/Code_Projects/pnf/data/nifty.csv')

In [2]:
with data_path.open(newline="") as handle:
    rows = list(DictReader(handle))

len(rows), rows[:3], rows[-3:]

(8659,
 [{'time': '1990-07-03',
   'open': '279.01999',
   'high': '279.01999',
   'low': '279.01999',
   'close': '279.01999'},
  {'time': '1990-07-05',
   'open': '284.04001',
   'high': '284.04001',
   'low': '284.04001',
   'close': '284.04001'},
  {'time': '1990-07-06',
   'open': '289.04001',
   'high': '289.04001',
   'low': '289.04001',
   'close': '289.04001'}],
 [{'time': '2026-04-08',
   'open': '23855.15',
   'high': '24025.15',
   'low': '23828.5',
   'close': '23997.35'},
  {'time': '2026-04-09',
   'open': '23909.05',
   'high': '23990.75',
   'low': '23682.8',
   'close': '23775.1'},
  {'time': '2026-04-10',
   'open': '23880.55',
   'high': '24074.05',
   'low': '23856.35',
   'close': '24050.6'}])

In [3]:
from pnf.core import PnFConfig, build_columns, calculate_logscale_box_size, calculate_step_box_size
from pnf.patterns import detect_patterns
from pnf.levels import detect_horizontal_levels
from pnf.counts import vertical_count
from pnf.strategy import generate_trades
from pnf.render import columns_to_table, render_ascii_chart

config = PnFConfig(box_pct=0.0025, reversal=3, method="close", scaling="step_box")
series = [{"close": row["close"]} for row in rows]
columns = build_columns(series, config)

last_reference = columns[-1].last_box
next_step_box_size = calculate_step_box_size(last_reference, config)
print("step_box close columns:", len(columns))
print("last reference:", last_reference)
print("next step_box size:", next_step_box_size)
columns_to_table(columns[-5:])

step_box close columns: 1961
last reference: 24025.23
next step_box size: 60.063075


[{'type': 'X',
  'box_count': 13,
  'high': Decimal('23287.86'),
  'low': Decimal('22605.46'),
  'start_index': 8648,
  'end_index': 8649},
 {'type': 'O',
  'box_count': 16,
  'high': Decimal('23229.64'),
  'low': Decimal('22365.66'),
  'start_index': 8650,
  'end_index': 8651},
 {'type': 'X',
  'box_count': 28,
  'high': Decimal('23967.11'),
  'low': Decimal('22421.57'),
  'start_index': 8652,
  'end_index': 8656},
 {'type': 'O',
  'box_count': 3,
  'high': Decimal('23907.19'),
  'low': Decimal('23787.36'),
  'start_index': 8657,
  'end_index': 8657},
 {'type': 'X',
  'box_count': 4,
  'high': Decimal('24025.23'),
  'low': Decimal('23846.83'),
  'start_index': 8658,
  'end_index': 8658}]

In [4]:
logscale_config = PnFConfig(box_pct=0.0025, reversal=3, method="close", scaling="log")
logscale_series = [{"close": row["close"]} for row in rows]
logscale_columns = build_columns(logscale_series, logscale_config)

last_reference = logscale_columns[-1].last_box
next_box_size = calculate_logscale_box_size(last_reference, logscale_config)
print("true logscale close columns:", len(logscale_columns))
print("last reference:", last_reference)
print("next logscale box size:", next_box_size)
columns_to_table(logscale_columns[-5:])

true logscale close columns: 1989
last reference: 24000.83
next logscale box size: 60.002075


[{'type': 'X',
  'box_count': 10,
  'high': Decimal('23767.82'),
  'low': Decimal('23239.67'),
  'start_index': 8642,
  'end_index': 8644},
 {'type': 'O',
  'box_count': 21,
  'high': Decimal('23708.40'),
  'low': Decimal('22550.71'),
  'start_index': 8645,
  'end_index': 8647},
 {'type': 'X',
  'box_count': 13,
  'high': Decimal('23294.71'),
  'low': Decimal('22607.09'),
  'start_index': 8648,
  'end_index': 8649},
 {'type': 'O',
  'box_count': 16,
  'high': Decimal('23236.47'),
  'low': Decimal('22380.18'),
  'start_index': 8650,
  'end_index': 8651},
 {'type': 'X',
  'box_count': 28,
  'high': Decimal('24000.83'),
  'low': Decimal('22436.13'),
  'start_index': 8652,
  'end_index': 8658}]

In [5]:
patterns = detect_patterns(columns, config)
levels = detect_horizontal_levels(columns, config, min_touches=3, cluster_threshold_boxes=1)
vcount = vertical_count(columns, config, patterns)
trades = generate_trades(columns, config, patterns=patterns, levels=levels)

{
    "rows": len(rows),
    "columns": len(columns),
    "patterns": len(patterns),
    "levels": len(levels),
    "vertical_count": vcount,
    "trades": len(trades),
}

{'rows': 8659,
 'columns': 1961,
 'patterns': 1464,
 'levels': 4,
 'vertical_count': CountTarget(kind='vertical', direction='bullish', start_column=2, end_column=2, count=76, box_size=Decimal('0.867175'), base_price=Decimal('346.87'), target=Decimal('544.59')),
 'trades': 2}

In [6]:
patterns[-10:]

[PatternSignal(name='double_bottom_breakdown', direction='bearish', column_index=1951, level=Decimal('25463.96'), breakout=Decimal('24519.41'), confirmed_by=(1949, 1950)),
 PatternSignal(name='bearish_catapult', direction='bearish', column_index=1953, level=Decimal('24519.41'), breakout=Decimal('23195.10'), confirmed_by=(1949, 1951, 1952)),
 PatternSignal(name='double_bottom_breakdown', direction='bearish', column_index=1953, level=Decimal('24519.41'), breakout=Decimal('23195.10'), confirmed_by=(1951, 1952)),
 PatternSignal(name='bearish_catapult', direction='bearish', column_index=1955, level=Decimal('23195.10'), breakout=Decimal('22549.09'), confirmed_by=(1951, 1953, 1954)),
 PatternSignal(name='double_bottom_breakdown', direction='bearish', column_index=1955, level=Decimal('23195.10'), breakout=Decimal('22549.09'), confirmed_by=(1953, 1954)),
 PatternSignal(name='bearish_catapult', direction='bearish', column_index=1957, level=Decimal('22549.09'), breakout=Decimal('22365.66'), confi

In [7]:
levels

[PriceLevel(kind='support', price=Decimal('2323.50'), box=14907, touches=3, column_indices=(897, 887, 885), start_column=885, end_column=897),
 PriceLevel(kind='resistance', price=Decimal('6074.60'), box=23000, touches=3, column_indices=(1402, 1346, 1344), start_column=1344, end_column=1402),
 PriceLevel(kind='resistance', price=Decimal('15845.99'), box=27645, touches=3, column_indices=(1754, 1752, 1750), start_column=1750, end_column=1754),
 PriceLevel(kind='resistance', price=Decimal('26209.98'), box=29926, touches=3, column_indices=(1938, 1894, 1942), start_column=1894, end_column=1942)]

In [8]:
trades[:10]

[TradeSignal(direction='bullish', entry=Decimal('346.87'), stop_loss=Decimal('288.00'), target=Decimal('544.59'), pattern_name='double_top_breakout', column_index=2, count_kind='vertical'),
 TradeSignal(direction='bearish', entry=Decimal('399.89'), stop_loss=Decimal('480.27'), target=Decimal('186.95'), pattern_name='double_bottom_breakdown', column_index=11, count_kind='vertical')]

In [9]:
print(render_ascii_chart(columns[-10:]))

         0 1 2 3 4 5 6 7 8 9
 25590.8 O . . . . . . . . .
25526.67 O . . . . . . . . .
25462.53 O . . . . . . . . .
25398.87 O . . . . . . . . .
25335.22 O . . . . . . . . .
25271.56 O . . . . . . . . .
 25207.9 O . . . . . . . . .
25144.88 O . . . . . . . . .
25081.86 O . . . . . . . . .
25018.84 O . . . . . . . . .
24955.82 O . . . . . . . . .
 24892.8 O . . . . . . . . .
24830.57 O . . . . . . . . .
24768.34 O . . . . . . . . .
 24764.6 . X . . . . . . . .
 24706.1 O . . . . . . . . .
24703.31 . X . . . . . . . .
24702.69 . . O . . . . . . .
24643.87 O . . . . . . . . .
24642.01 . X . . . . . . . .
24640.78 . . O . . . . . . .
24581.64 O . . . . . . . . .
24580.71 . X . . . . . . . .
24578.87 . . O . . . . . . .
24519.41 O . . . . . . . . .
24516.95 . . O . . . . . . .
24455.04 . . O . . . . . . .
 24393.9 . . O . . . . . . .
24332.76 . . O . . . . . . .
24271.63 . . O . . . . . . .
24210.49 . . O . . . . . . .
24149.35 . . O . . . . . . .
24088.21 . . O . . . . . . .
24027.99 . . O

In [10]:
config_hl = PnFConfig(box_pct=0.0025, reversal=3, method="high_low", scaling="step_box")
hl_series = [{"high": row["high"], "low": row["low"]} for row in rows]
hl_columns = build_columns(hl_series, config_hl)

print("step_box high_low columns:", len(hl_columns))
columns_to_table(hl_columns[-5:])

step_box high_low columns: 3419


[{'type': 'X',
  'box_count': 11,
  'high': Decimal('22909.96'),
  'low': Decimal('22352.54'),
  'start_index': 8652,
  'end_index': 8652},
 {'type': 'O',
  'box_count': 12,
  'high': Decimal('22852.69'),
  'low': Decimal('22222.66'),
  'start_index': 8653,
  'end_index': 8653},
 {'type': 'X',
  'box_count': 31,
  'high': Decimal('23983.88'),
  'low': Decimal('22278.22'),
  'start_index': 8654,
  'end_index': 8656},
 {'type': 'O',
  'box_count': 5,
  'high': Decimal('23923.92'),
  'low': Decimal('23684.08'),
  'start_index': 8657,
  'end_index': 8657},
 {'type': 'X',
  'box_count': 6,
  'high': Decimal('24039.34'),
  'low': Decimal('23743.29'),
  'start_index': 8658,
  'end_index': 8658}]

In [11]:
config_close = PnFConfig(box_pct=0.0025, reversal=3, method="close", scaling="step_box")
close_series = [{"close": row["close"]} for row in rows]
close_columns = build_columns(close_series, config_close)

print("step_box close columns:", len(close_columns))
columns_to_table(close_columns[-10:])

step_box close columns: 1961


[{'type': 'O',
  'box_count': 18,
  'high': Decimal('25590.80'),
  'low': Decimal('24519.41'),
  'start_index': 8629,
  'end_index': 8634},
 {'type': 'X',
  'box_count': 4,
  'high': Decimal('24764.60'),
  'low': Decimal('24580.71'),
  'start_index': 8635,
  'end_index': 8635},
 {'type': 'O',
  'box_count': 26,
  'high': Decimal('24702.69'),
  'low': Decimal('23195.10'),
  'start_index': 8636,
  'end_index': 8641},
 {'type': 'X',
  'box_count': 9,
  'high': Decimal('23720.91'),
  'low': Decimal('23253.09'),
  'start_index': 8642,
  'end_index': 8644},
 {'type': 'O',
  'box_count': 20,
  'high': Decimal('23661.61'),
  'low': Decimal('22549.09'),
  'start_index': 8645,
  'end_index': 8647},
 {'type': 'X',
  'box_count': 13,
  'high': Decimal('23287.86'),
  'low': Decimal('22605.46'),
  'start_index': 8648,
  'end_index': 8649},
 {'type': 'O',
  'box_count': 16,
  'high': Decimal('23229.64'),
  'low': Decimal('22365.66'),
  'start_index': 8650,
  'end_index': 8651},
 {'type': 'X',
  'box_

In [12]:
config_hlc = PnFConfig(box_pct=0.0025, reversal=3, method="high_low_close", scaling="step_box")
hlc_series = [{"high": row["high"], "low": row["low"], "close": row["close"]} for row in rows]
hlc_columns = build_columns(hlc_series, config_hlc)

print("step_box high_low_close columns:", len(hlc_columns))
columns_to_table(hlc_columns[-5:])

step_box high_low_close columns: 3419


[{'type': 'X',
  'box_count': 11,
  'high': Decimal('22909.96'),
  'low': Decimal('22352.54'),
  'start_index': 8652,
  'end_index': 8652},
 {'type': 'O',
  'box_count': 12,
  'high': Decimal('22852.69'),
  'low': Decimal('22222.66'),
  'start_index': 8653,
  'end_index': 8653},
 {'type': 'X',
  'box_count': 31,
  'high': Decimal('23983.88'),
  'low': Decimal('22278.22'),
  'start_index': 8654,
  'end_index': 8656},
 {'type': 'O',
  'box_count': 5,
  'high': Decimal('23923.92'),
  'low': Decimal('23684.08'),
  'start_index': 8657,
  'end_index': 8657},
 {'type': 'X',
  'box_count': 6,
  'high': Decimal('24039.34'),
  'low': Decimal('23743.29'),
  'start_index': 8658,
  'end_index': 8658}]

## All Price Series x Box Sizing Examples

This section runs all three price input methods with both box sizing modes.

- `close`: close price only
- `high_low`: high and low only
- `high_low_close`: high, low, and close
- `step_box`: frozen percentage box size per update
- `log`: true compound/log box sizing, box size changes after every box


In [13]:
def make_series(method, rows):
    if method == "close":
        return [{"close": row["close"]} for row in rows]
    if method == "high_low":
        return [{"high": row["high"], "low": row["low"]} for row in rows]
    if method == "high_low_close":
        return [{"high": row["high"], "low": row["low"], "close": row["close"]} for row in rows]
    raise ValueError(method)

examples = {}
summary = []
for method in ("close", "high_low", "high_low_close"):
    for scaling in ("step_box", "log"):
        cfg = PnFConfig(box_pct=0.0025, reversal=3, method=method, scaling=scaling)
        cols = build_columns(make_series(method, rows), cfg)
        helper = calculate_step_box_size if scaling == "step_box" else calculate_logscale_box_size
        next_box_size = helper(cols[-1].last_box, cfg) if cols else None
        key = f"{method}_{scaling}"
        examples[key] = {"config": cfg, "columns": cols, "next_box_size": next_box_size}
        summary.append({
            "series": method,
            "box_sizing": scaling,
            "columns": len(cols),
            "last_type": cols[-1].type if cols else None,
            "last_box_count": cols[-1].box_count if cols else None,
            "last_reference": cols[-1].last_box if cols else None,
            "next_box_size": next_box_size,
        })

summary

[{'series': 'close',
  'box_sizing': 'step_box',
  'columns': 1961,
  'last_type': 'X',
  'last_box_count': 4,
  'last_reference': Decimal('24025.23'),
  'next_box_size': Decimal('60.063075')},
 {'series': 'close',
  'box_sizing': 'log',
  'columns': 1989,
  'last_type': 'X',
  'last_box_count': 28,
  'last_reference': Decimal('24000.83'),
  'next_box_size': Decimal('60.002075')},
 {'series': 'high_low',
  'box_sizing': 'step_box',
  'columns': 3419,
  'last_type': 'X',
  'last_box_count': 6,
  'last_reference': Decimal('24039.34'),
  'next_box_size': Decimal('60.098350')},
 {'series': 'high_low',
  'box_sizing': 'log',
  'columns': 3377,
  'last_type': 'X',
  'last_box_count': 5,
  'last_reference': Decimal('24016.07'),
  'next_box_size': Decimal('60.040175')},
 {'series': 'high_low_close',
  'box_sizing': 'step_box',
  'columns': 3419,
  'last_type': 'X',
  'last_box_count': 6,
  'last_reference': Decimal('24039.34'),
  'next_box_size': Decimal('60.098350')},
 {'series': 'high_low_cl

In [14]:
for key, item in examples.items():
    print("\n", key)
    print("next_box_size:", item["next_box_size"])
    for row in columns_to_table(item["columns"][-3:]):
        print(row)


 close_step_box
next_box_size: 60.063075
{'type': 'X', 'box_count': 28, 'high': Decimal('23967.11'), 'low': Decimal('22421.57'), 'start_index': 8652, 'end_index': 8656}
{'type': 'O', 'box_count': 3, 'high': Decimal('23907.19'), 'low': Decimal('23787.36'), 'start_index': 8657, 'end_index': 8657}
{'type': 'X', 'box_count': 4, 'high': Decimal('24025.23'), 'low': Decimal('23846.83'), 'start_index': 8658, 'end_index': 8658}

 close_log
next_box_size: 60.002075
{'type': 'X', 'box_count': 13, 'high': Decimal('23294.71'), 'low': Decimal('22607.09'), 'start_index': 8648, 'end_index': 8649}
{'type': 'O', 'box_count': 16, 'high': Decimal('23236.47'), 'low': Decimal('22380.18'), 'start_index': 8650, 'end_index': 8651}
{'type': 'X', 'box_count': 28, 'high': Decimal('24000.83'), 'low': Decimal('22436.13'), 'start_index': 8652, 'end_index': 8658}

 high_low_step_box
next_box_size: 60.098350
{'type': 'X', 'box_count': 31, 'high': Decimal('23983.88'), 'low': Decimal('22278.22'), 'start_index': 8654, '

In [15]:
example_key = "close_log"
print(example_key)
print(render_ascii_chart(examples[example_key]["columns"][-6:]))

close_log
         0 1 2 3 4 5
24189.45 O . . . . .
24128.98 O . . . . .
24068.66 O . . . . .
24008.49 O . . . . .
24000.83 . . . . . X
23948.47 O . . . . .
23940.98 . . . . . X
 23888.6 O . . . . .
23881.28 . . . . . X
23828.88 O . . . . .
23821.73 . . . . . X
23769.31 O . . . . .
23767.82 . X . . . .
23762.32 . . . . . X
23709.89 O . . . . .
23708.55 . X . . . .
 23708.4 . . O . . .
23703.06 . . . . . X
23650.62 O . . . . .
23649.43 . X . . . .
23649.13 . . O . . .
23643.95 . . . . . X
23591.49 O . . . . .
23590.45 . X . . . .
23590.01 . . O . . .
23584.99 . . . . . X
23532.51 O . . . . .
23531.62 . X . . . .
23531.03 . . O . . .
23526.17 . . . . . X
23473.68 O . . . . .
23472.94 . X . . . .
 23472.2 . . O . . .
 23467.5 . . . . . X
   23415 O . . . . .
 23414.4 . X . . . .
23413.52 . . O . . .
23408.98 . . . . . X
23356.46 O . . . . .
23356.01 . X . . . .
23354.99 . . O . . .
 23350.6 . . . . . X
23298.07 O . . . . .
23297.77 . X . . . .
 23296.6 . . O . . .
23294.71 . . . X . .
232